# 🔹 Cell 1: Import Libraries and Set Seed
Import necessary libraries and set the random seed for reproducibility.

In [14]:
import torch
import pandas as pd
from transformers import set_seed, GPT2Tokenizer, GPT2LMHeadModel
from bertviz import model_view
set_seed(42)

# 🔹 Cell 2: Load GPT-2 Model and Tokenizer
Load the pre-trained GPT-2 model and tokenizer. 
Enable attention outputs for visualization.

In [9]:
# 1. Load Model and Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained(
    'gpt2', 
    output_attentions=True, 
    attn_implementation="eager"
)

The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

# 🔹 Cell 3: Encode Input Phrase
Convert input text into token IDs and get the token strings for later use.

In [10]:
# 2. Encode input
phrase = "My friend was right about this class. It is so fun!"
encoded_phrase = tokenizer(phrase, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(encoded_phrase['input_ids'][0])
tokens

['My',
 'Ġfriend',
 'Ġwas',
 'Ġright',
 'Ġabout',
 'Ġthis',
 'Ġclass',
 '.',
 'ĠIt',
 'Ġis',
 'Ġso',
 'Ġfun',
 '!']

# 🔹 Cell 4: Get Model Outputs with Attentions and Hidden States
Run the model to get attention scores and hidden states.

In [11]:
response = model(**encoded_phrase, output_attentions=True, output_hidden_states=True)
print(len(response.attentions))

12


# 🔹 Cell 5: Visualize Attention for Layer 9, Head 0
Create a DataFrame showing attention scores for a specific layer and head.

In [13]:
# Layer index 9, head 0. Check out the almost 60% attention the token it is giving to the token class
arr = response.attentions[9][0][0]

n_digits = 3

attention_df = pd.DataFrame((torch.round(arr * 10**n_digits) / (10**n_digits)).detach()).applymap(float)

attention_df.columns = tokens
attention_df.index = tokens

attention_df


/var/folders/c4/_8nhzl6j7150sh__f3lw818c0000gn/T/ipykernel_11177/1658672087.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  attention_df = pd.DataFrame((torch.round(arr * 10**n_digits) / (10**n_digits)).detach()).applymap(float)


,My,Ġfriend,Ġwas,Ġright,Ġabout,Ġthis,Ġclass,.,ĠIt,Ġis,Ġso,Ġfun,!
My,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Ġfriend,0.968,0.032,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Ġwas,0.824,0.145,0.031,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Ġright,0.979,0.008,0.007,0.005,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Ġabout,0.979,0.008,0.004,0.005,0.005,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Ġthis,0.924,0.031,0.007,0.006,0.016,0.016,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Ġclass,0.946,0.005,0.001,0.001,0.001,0.002,0.044,0.000,0.000,0.000,0.000,0.000,0.000
.,0.691,0.013,0.003,0.003,0.002,0.006,0.269,0.013,0.000,0.000,0.000,0.000,0.000
ĠIt,0.318,0.003,0.003,0.003,0.006,0.018,0.599,0.018,0.032,0.000,0.000,0.000,0.000
Ġis,0.331,0.006,0.002,0.002,0.003,0.018,0.533,0.013,0.062,0.030,0.000,0.000,0.000


# 🔹 Cell 6: Visualize Attention Across All Layers
Use BertViz to visualize attention for all layers and heads.

In [15]:
tokens = tokenizer.convert_ids_to_tokens(encoded_phrase['input_ids'][0]) 
model_view(response.attentions, tokens)

<IPython.core.display.Javascript object>

# 🔹 Cell 7: Inspect Hidden States and Logits Shapes
Check the shape of the last hidden state and model logits.

In [16]:
response.hidden_states[-1].shape

torch.Size([1, 13, 768])

In [17]:
response.logits.shape

torch.Size([1, 13, 50257])

# 🔹 Cell 8: Map Tokens to Predicted Next Tokens
Create a DataFrame pairing each token with the model's predicted next token.

In [18]:
pd.DataFrame(
    zip(tokens, tokenizer.convert_ids_to_tokens(response.logits.argmax(2)[0])), 
    columns=['Sequence up until', 'Next token with highest probability']
)

,Sequence up until,Next token with highest probability
0,My,Ċ
1,Ġfriend,","
2,Ġwas,Ġa
3,Ġright,.
4,Ġabout,Ġthat
5,Ġthis,.
6,Ġclass,.
7,.,ĠI
8,ĠIt,'s
9,Ġis,Ġa
